# Loading suite2p output with `Suite2pImaging`

This notebook shows how to load a **pre-computed suite2p run** — the registered
binary movies suite2p writes to disk — into `photon-mosaic`, following the three
use cases from [issue #77](https://github.com/photon-mosaic/photon-mosaic/issues/77):

1. **Per-plane load + stitch** — each plane is a `Suite2pImaging`; combine them with `concatenate_planes`.
2. **Whole folder** — `read_suite2p` loads every plane, stitches them, and marks the movie as registered (one object per functional channel).
3. **Per-file epochs** — `split_suite2p_into_files` recovers one epoch per source acquisition file from `ops['frames_per_file']`.

Point `SUITE2P_FOLDER` below at your own suite2p output root: the parent of
`plane0/`, `plane1/`, … , each containing `data.bin` and `ops.npy`.

## Setup

Edit the path to point at your suite2p output folder.

In [ ]:
%matplotlib widget
from pathlib import Path

from photon_mosaic.core import concatenate_planes
from photon_mosaic.extractors.suite2p import (
    Suite2pImaging,
    read_suite2p,
    split_suite2p_into_files,
)
from photon_mosaic.widgets.series import plot_imaging_series

# EDIT ME: a suite2p output root (the parent of plane0/, plane1/, ...)
SUITE2P_FOLDER = Path("your/path/to/suite2p/precomputed/files")

## Use case 1 — load each plane, then stitch

Each plane directory is loaded as its own single-plane `Suite2pImaging`. Because a
single plane maps to one binary on disk, these objects stay binary-compatible.
`concatenate_planes` then stacks them into one `(T, H, W, n_planes)` volume — a lazy
view that pulls pixels from each plane's memmap on read, without copying.

In [ ]:
plane0 = Suite2pImaging(SUITE2P_FOLDER / "plane0")
plane1 = Suite2pImaging(SUITE2P_FOLDER / "plane1")
print("plane0 shape:", plane0.shape, "| registered:", plane0.is_registered)

volume = concatenate_planes(plane0, plane1)
print("stitched volume shape:", volume.shape)

## Use case 2 — load a whole folder with `read_suite2p`

`read_suite2p` finds every `plane*/` under the folder, loads each as a
`Suite2pImaging`, and stitches them for you. It returns a single object for a
one-channel run (a `(chan1, chan2)` tuple when the run has two functional
channels), with `is_registered=True`.

In [ ]:
imaging = read_suite2p(SUITE2P_FOLDER)
if isinstance(imaging, tuple):  # two functional channels
    imaging, imaging_chan2 = imaging

print("shape        :", imaging.shape)
print("num planes   :", imaging.num_planes)
print("sampling rate:", imaging.sampling_frequency, "Hz")
print("is_registered:", imaging.is_registered)

plot_imaging_series(imaging, backend="ipywidgets", colormap="plasma")

## Use case 3 — split into one epoch per source file

suite2p concatenates the source acquisition files into one registered movie and
records the per-file frame counts in `ops['frames_per_file']`. `photon-mosaic`
keeps those counts on the imaging object, so `split_suite2p_into_files` can turn
the single long movie back into one epoch per original file — lazily, without
re-reading any pixels.

In [ ]:
per_file = split_suite2p_into_files(imaging)
print("source files (epochs):", per_file.get_num_epochs())
sizes = [per_file.get_num_samples(segment_index=i) for i in range(per_file.get_num_epochs())]
print("frames per file      :", sizes)